In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('../../src'))

from traffic_normalization import *
from hierarchical_seasonal_analysis import AccidentDataProcessor
from population_normalization import AccidentNormalizer



/opt/anaconda3/envs/new_Bachelorarbeit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

# Normalizaiton Population and Regions for all seasons Germany

In [3]:
seasons = [["3","4","5"],["6","7","8"],["9","10","11"],["12","1","2"]]

different_months_ger = []
seasons_ger =[]

# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/germany/geofiles/Germany_merged.geojson'
regions_gdf = gpd.read_file(region_path)
for i in seasons:
    # 2. Initialize Normalizer
    normalizer = AccidentNormalizer(regions_gdf)
    # 3. Process Accidents
    acc_path = '../../data/preprocessed/germany/collisions/preprocessed_ger.csv'

    acc_gdf = normalizer.load_accident_data(acc_path, filters={"month":i,"casualty_severity": ["1"]})

    # 4. Generate Final Dataset
    final_regions = normalizer.attach_regions_and_count(acc_gdf)
    final_regions = normalizer.normalize_by_population(scale=100_000)

    seasons_ger.append(final_regions)
    

    

In [4]:
mean_ger_spring = seasons_ger[0]["accidents_per_100k"].mean()
print("mean spring",mean_ger_spring)
mean_ger_summer = seasons_ger[1]["accidents_per_100k"].mean()
print("mean summer",mean_ger_summer)
mean_ger_autumn = seasons_ger[2]["accidents_per_100k"].mean()
print("mean autumn",mean_ger_autumn)
mean_ger_winter = seasons_ger[3]["accidents_per_100k"].mean()
print("mean winter",mean_ger_winter)

print(mean_ger_spring + mean_ger_summer + mean_ger_autumn + mean_ger_winter)

mean spring 0.7192610106398576
mean summer 0.9277363103810313
mean autumn 0.7402677414394172
mean winter 0.5967663056244426
2.9840313680847483


# Normalizaiton Population and Regions for the year Germany

In [5]:
# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/germany/geofiles/Germany_merged.geojson'
regions_gdf = gpd.read_file(region_path)

# 2. Initialize Normalizer
normalizer = AccidentNormalizer(regions_gdf)
# 3. Process Accidents
acc_path = '../../data/preprocessed/germany/collisions/preprocessed_ger.csv'
acc_gdf = normalizer.load_accident_data(acc_path, filters={"casualty_severity": ["1"]})
# 4. Generate Final Dataset
final_regions_ger = normalizer.attach_regions_and_count(acc_gdf)
final_regions_ger = normalizer.normalize_by_population(scale=100_000)

mean_ger_all = final_regions_ger["accidents_per_100k"].mean()
print(mean_ger_all)


2.984031368084749


# Normalizaiton Population and Regions for all seasons UK

In [6]:
seasons = [["3","4","5"],["6","7","8"],["9","10","11"],["12","1","2"]]

seasons_uk = []

# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/uk/geofiles/UK_merged.geojson'
regions_gdf = gpd.read_file(region_path)
for i in seasons:
    # 2. Initialize Normalizer
    normalizer = AccidentNormalizer(regions_gdf)
    # 3. Process Accidents
    acc_path = '../../data/preprocessed/uk/collisions/preprocessed_uk.csv'

    acc_gdf = normalizer.load_accident_data(acc_path, filters={"month":i,"casualty_severity": ["1"]})

    # 4. Generate Final Dataset
    final_regions_seasons_uk = normalizer.attach_regions_and_count(acc_gdf)
    final_regions_seasons_uk = normalizer.normalize_by_population(scale=100_000)
    # 5. Diagnostic Plot: Check if the map looks correct (no empty regions where expected)
    seasons_uk.append(final_regions_seasons_uk)
    

In [7]:
mean_uk_spring = seasons_uk[0]["accidents_per_100k"].mean()
print("mean spring uk",mean_uk_spring)
mean_uk_summer = seasons_uk[1]["accidents_per_100k"].mean()
print("mean summer uk",mean_uk_summer)
mean_uk_autumn = seasons_uk[2]["accidents_per_100k"].mean()
print("mean autumn uk",mean_uk_autumn)
mean_uk_winter = seasons_uk[3]["accidents_per_100k"].mean()
print("mean winter uk",mean_uk_winter)
print(mean_uk_spring+mean_uk_summer+mean_uk_autumn+mean_uk_winter)

mean spring uk 0.5211649053479659
mean summer uk 0.595469060958178
mean autumn uk 0.5170101500954623
mean winter uk 0.5312407518592461
2.1648848682608524


# Normalizaiton Population and Regions for the year UK

In [8]:
# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/uk/geofiles/UK_merged.geojson'
regions_gdf = gpd.read_file(region_path)

# 2. Initialize Normalizer
normalizer = AccidentNormalizer(regions_gdf)
# 3. Process Accidents
#acc_path = '../data/processed/reduced_uk_dataset.csv'
acc_gdf = normalizer.load_accident_data(acc_path, filters={"casualty_severity": ["1"]})
# 4. Generate Final Dataset
final_regions_uk = normalizer.attach_regions_and_count(acc_gdf)
final_regions_uk = normalizer.normalize_by_population(scale=100_000)

mean_uk_all = final_regions_uk["accidents_per_100k"].mean()
print(mean_uk_all)

2.1648848682608524


# AADF Normalization Germany

In [9]:
proc = AccidentDataProcessor()

# Germany
ger_regions = gpd.read_file('../../data/preprocessed/germany/geofiles/ger_gdf_with_osm_roads.gpkg')
ger_acc = proc.load_accidents('../../data/preprocessed/germany/collisions/preprocessed_ger.csv',
                              category_filters={"casualty_severity": [1]})

ger_merged = proc.aggregate_by_region_monthly(ger_regions, ger_acc)
ger_traffic_exposure_gdf = process_country(ger_merged, 'Germany')
ger_normalized_by_traffic = normalize_accident_count(ger_traffic_exposure_gdf, 1e9)
traffic_ger = ger_normalized_by_traffic["accident_rate"].mean()
print(traffic_ger)


2.9787809939484555


In [10]:
seasons = [[3,4,5],[6,7,8],[9,10,11],[12,1,2]]
seasons_ger_aadf= []
for i in seasons:
    season_g =  ger_normalized_by_traffic[ger_normalized_by_traffic['month'].isin(i)]
    mean = season_g["accident_rate"].mean()
    seasons_ger_aadf.append(mean)

means_ger = []
for i in seasons_ger_aadf:
    means_ger.append(i)
    print(i)


2.910677269635297
3.4074388691884825
3.2036081235769585
2.3933997133930833


# AADF Normalization UK

In [11]:
proc = AccidentDataProcessor()

# Germany
uk_regions = gpd.read_file('../../data/preprocessed/uk/geofiles/uk_gdf_with_osm_roads.gpkg')
uk_acc = proc.load_accidents('../../data/preprocessed/uk/collisions/preprocessed_uk.csv',
                              category_filters={"casualty_severity": [1]})

uk_merged = proc.aggregate_by_region_monthly(uk_regions, uk_acc)
uk_traffic_exposure_gdf = process_country(uk_merged, 'UK')
uk_normalized_by_traffic = normalize_accident_count(uk_traffic_exposure_gdf, 1e9)
#print(uk_normalized_by_traffic)
traffic_uk = uk_normalized_by_traffic["accident_rate"].mean()
print(traffic_uk)

1.7292904903625974


In [12]:
seasons = [[3,4,5],[6,7,8],[9,10,11],[12,1,2]]
seasons_uk_aadf= []
for i in seasons:
    season_uk =  uk_normalized_by_traffic[uk_normalized_by_traffic['month'].isin(i)]
    mean = season_uk["accident_rate"].mean()
    seasons_uk_aadf.append(mean)

means_uk = []
for i in seasons_uk_aadf:
    means_uk.append(i)
    print(i)


1.686588257114163
1.846589757949043
1.723177432588794
1.6608065137983896


# Normalization with population Germany

In [13]:
seasons_str = [["3","4","5"],["6","7","8"],["9","10","11"],["12","1","2"]]

seasons_ger_full =[]


# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/germany/geofiles/Germany_full.geojson'
regions_gdf = gpd.read_file(region_path)
for i in seasons_str:
    # 2. Initialize Normalizer
    normalizer = AccidentNormalizer(regions_gdf)
    # 3. Process Accidents
    acc_path = '../../data/preprocessed/germany/collisions/preprocessed_ger.csv'

    acc_gdf = normalizer.load_accident_data(acc_path, filters={"month":i,"casualty_severity": ["1"]})

    # 4. Generate Final Dataset
    final_regions_seasons_ger_full = normalizer.attach_regions_and_count(acc_gdf)
    final_regions_seasons_ger_full = normalizer.normalize_by_population(scale=100_000)

    seasons_ger_full.append(final_regions_seasons_ger_full)

In [14]:
mean_ger_spring_pop = seasons_ger_full[0]["accidents_per_100k"].mean()
print("mean spring",mean_ger_spring_pop)
mean_ger_summer_pop = seasons_ger_full[1]["accidents_per_100k"].mean()
print("mean summer",mean_ger_summer_pop)
mean_ger_autumn_pop = seasons_ger_full[2]["accidents_per_100k"].mean()
print("mean autumn",mean_ger_autumn_pop)
mean_ger_winter_pop = seasons_ger_full[3]["accidents_per_100k"].mean()
print("mean winter",mean_ger_winter_pop)

mean spring 0.6788821000349778
mean summer 1.179744359291363
mean autumn 0.7924623247535996
mean winter 0.6270219343524172


In [15]:
# 1. Load your 500k-merged regions
region_path = '../../data/preprocessed/germany/geofiles/Germany_full.geojson'
regions_gdf = gpd.read_file(region_path)

# 2. Initialize Normalizer
normalizer = AccidentNormalizer(regions_gdf)
# 3. Process Accidents
acc_path = '../../data/preprocessed/germany/collisions/preprocessed_ger.csv'
acc_gdf = normalizer.load_accident_data(acc_path, filters={"casualty_severity": ["1"]})
# 4. Generate Final Dataset
final_regions_full = normalizer.attach_regions_and_count(acc_gdf)
final_regions_full = normalizer.normalize_by_population(scale=100_000)

mean_ger_all_pop = final_regions_full["accidents_per_100k"].mean()
print(mean_ger_all_pop)


3.2781107184323575


In [16]:
data_different_normazations = {
    "mean GER spring (regions and population)": [mean_ger_spring],
    "mean GER summer (regions and population)": [mean_ger_summer],
    "mean GER autumn (regions and population)": [mean_ger_autumn],
    "mean GER winter (regions and population)": [mean_ger_winter],
    "mean GER all (regions and population)": [mean_ger_all],
    "mean UK spring (regions and population)": [mean_uk_spring],
    "mean UK summer (regions and population)": [mean_uk_summer],
    "mean UK autumn (regions and population)": [mean_uk_autumn],
    "mean UK winter (regions and population)": [mean_uk_winter],
    "mean UK all (regions and population)": [mean_uk_all],
    "mean GER spring (traffic)": [means_ger[0]],
    "mean GER summer (traffic)": [means_ger[1]],
    "mean GER autumn (traffic)": [means_ger[2]],
    "mean GER winter (traffic)": [means_ger[3]],
    "mean GER all (traffic)": [traffic_ger],
    "mean UK spring (traffic)": [means_uk[0]],
    "mean UK summer (traffic)": [means_uk[1]],
    "mean UK autumn (traffic)": [means_uk[2]],
    "mean UK winter (traffic)": [means_uk[3]]
}

df_normalizaions = pd.DataFrame([
    {"name": key, "value": value[0]}
    for key, value in data_different_normazations.items()
])

df_normalizaions.to_csv('../../results/values_normalization/normalzaion_values', index=False)
df_normalizaions


,name,value
0,mean GER spring (regions and population),0.719261
1,mean GER summer (regions and population),0.927736
2,mean GER autumn (regions and population),0.740268
3,mean GER winter (regions and population),0.596766
4,mean GER all (regions and population),2.984031
5,mean UK spring (regions and population),0.521165
6,mean UK summer (regions and population),0.595469
7,mean UK autumn (regions and population),0.517010
8,mean UK winter (regions and population),0.531241
9,mean UK all (regions and population),2.164885
